In [1]:
%useLatestDescriptors
%use datetime
%use dataframe
%use kandy

In [2]:
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern

val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class)
val currentTime = now
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH:mm:ss")})

@OptIn(FormatStringsInDatetimeFormats::class)
val previous24Hour = now
    .minus(24, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd")})

print("Current time : ${currentTime}, Previous time : ${previous24Hour}")

Current time : 2026-01-25 13:18:58, Previous time : 2026-01-24

In [3]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/full-stack-task-manager/client/src/main/resources/OceanWaterQuality.json"
val numOfRows = "1000"
val maxPage = 500

In [4]:
val serviceInfo = DataRow.readJson(path=serviceKeyFilePath)

In [5]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets
val url = "${serviceInfo.endpoint_xml}/${serviceInfo.rtmObservationInfo}?ServiceKey=${serviceInfo.key}&numOfRows=${numOfRows}&wtch_dt_start=${URLEncoder.encode(previous24Hour, StandardCharsets.UTF_8.toString())}"


In [6]:
USE{
    dependencies("org.json:json:20250107")
}

In [7]:
import org.json.XML

fun loadData(path:String, maxPage:Int): List<DataFrame<*>> {

    val rows = mutableListOf<DataFrame<*>>()
    var requestPage = 1
    do{
        val pagePath = "$path&pageNo=$requestPage"
        val jsonData = XML.toJSONObject(DataFrame.read(pagePath).toCsvStr())
        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
        try {
            val instanceDf = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
            requestPage += 1
            rows.add(instanceDf)
        } catch(e: Exception) {
            print(e.localizedMessage)
            break
        }
    } while (requestPage < maxPage )
    return rows
}

val dataList = loadData(url, maxPage)

Column not found: 'items'

In [8]:
val df = dataList.concat()
df.head(5)

rtmWqDoxn,rtmWqChpla,rtmWqBgalgsQy,rtmWqWtchStaCd,num,rtmWqTu,ph,rtmWqSlnty,rtmWqCndctv,rtmWqWtchDtlDt,rtmWtchWtem
13.866000,0.844000,,NEP2001,1,2,8.160000,9.549000,16.315001,2026-01-24 00:00:00.0,4.060000
10.560000,0.730000,,SEA5001,2,3,5.060000,24.627001,23.377001,2026-01-24 00:00:00.0,4.830000
11.530000,7.780000,,SEA1301,3,43,8.030000,33.522999,51.070000,2026-01-24 00:00:00.0,3.120000
9.060000,1.060000,,SEA7002,4,1,7.750000,31.347000,34.895000,2026-01-24 00:00:00.0,10.460000
10.990000,,,SEA1005,5,5,5.010000,0.672000,1.211000,2026-01-24 00:00:00.0,20.000000


item {
num string 순번
rtmWqWtchStaCd string 실시간수질관측정점코드
rtmWqWtchDtlDt string 실시간수질관측상세일시
rtmWtchWtem string 실시간관측수온
rtmWqCndctv string 실시간수질전기전도도
ph string 수소이온농도
rtmWqDoxn string 실시간수질용존산소량
rtmWqTu string 실시간수질탁도
rtmWqBgalgsQy string 실시간수질남조류량
rtmWqChpla string 실시간수질클로로필
rtmWqSlnty string 실시간수질염분
}

In [9]:
val url = "/Users/unchil/AndroidStudioProjects/full-stack-task-manager/client/src/main/resources/실시간 해양수질자동측정망 정점정보.csv"
val df_StaInfo = DataFrame.readCsv(url)
df_StaInfo.head(5)

해역구분,정점명,정점코드,경도,위도
특별관리해역,시화조력,SEA1002,126.611323,37.310004
특별관리해역,시화반월,SEA1005,126.823460,37.291451
특별관리해역,인천송도,SEA1006,126.626380,37.344882
특별관리해역,인천강화,SEA1007,126.526079,37.730790
특별관리해역,마산삼귀,SEA2004,128.595605,35.170995


In [10]:
val df_data = df.innerJoinWith(df_StaInfo){
    right.getValue<String>("정점코드") == rtmWqWtchStaCd
}
df_data.head(5)

rtmWqDoxn,rtmWqChpla,rtmWqBgalgsQy,rtmWqWtchStaCd,num,rtmWqTu,ph,rtmWqSlnty,rtmWqCndctv,rtmWqWtchDtlDt,rtmWtchWtem,해역구분,정점명,정점코드,경도,위도
13.866000,0.844000,,NEP2001,1,2,8.160000,9.549000,16.315001,2026-01-24 00:00:00.0,4.060000,하구 및 만,영산영암,NEP2001,126.448889,34.781667
10.560000,0.730000,,SEA5001,2,3,5.060000,24.627001,23.377001,2026-01-24 00:00:00.0,4.830000,특별관리해역,광양망덕,SEA5001,127.758468,34.968349
11.530000,7.780000,,SEA1301,3,43,8.030000,33.522999,51.070000,2026-01-24 00:00:00.0,3.120000,하구 및 만,천수만,SEA1301,126.456473,35.801637
9.060000,1.060000,,SEA7002,4,1,7.750000,31.347000,34.895000,2026-01-24 00:00:00.0,10.460000,특별관리해역,울산매암,SEA7002,129.387307,35.501987
10.990000,,,SEA1005,5,5,5.010000,0.672000,1.211000,2026-01-24 00:00:00.0,20.000000,특별관리해역,시화반월,SEA1005,126.823460,37.291451


In [11]:
df_data.select{ 정점명 and rtmWtchWtem }
    .groupBy{정점명}
    .sortBy { 정점명 }
    .plot{
        layout {
            title = "관측지점별 일평균 해수 온도 정보"
            size = 2000 to 1000
            //    theme = Theme.HIGH_CONTRAST_DARK
        }
        y.axis.limits = 10.0..25.0
        boxplot("정점명", "rtmWtchWtem") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("정점명"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="QPlE4g"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2000.0, 
 height: 1000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("QPlE4g");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"관측지점별 일평균 해수 온도 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[10.0,25.0]
},
"data":{
},
"ggsize":{
"width":2000.0,
"height":1000.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"정점명",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"rtmWtchWtem",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"정점명",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["광양망덕","광양적량","광양초남","금강하구","낙동명지","낙동을숙","마산봉암","새만금","시화반월","영산목포","영산영암","울산매암","인천송도","천수만"],
"정점명":["광양망덕","광양적량","광양초남","금강하구","낙동명지","낙동을숙","마산봉암","새만금","시화반월","영산목포","영산영암","울산매암","인천송도","천수만"],
"min":[3.4200000762939453,5.590000152587891,2.569999933242798,2.25,5.769999980926514,4.730000019073486,4.340000152587891,2.930000066757202,19.459999084472656,4.989999771118164,1.6799999475479126,9.850000381469727,1.1399999856948853,1.9700000286102295],
"middle":[4.539999961853027,6.090000152587891,4.980000019073486,2.5,6.994999885559082,6.384999990463257,5.090000152587891,3.450000047683716,19.989999771118164,5.519999980926514,3.4600000381469727,10.479999542236328,1.6799999475479126,2.7100000381469727],
"max":[6.309999942779541,6.820000171661377,7.420000076293945,2.7699999809265137,7.71999979019165,7.920000076293945,6.079999923706055,3.9100000858306885,20.959999084472656,5.880000114440918,4.869999885559082,11.170000076293945,2.119999885559082,3.240000009536743],
"lower":[4.099999904632568,5.827499985694885,3.950000047683716,2.427500069141388,6.362500071525574,5.929999828338623,4.7850000858306885,3.240000009536743,19.799999237060547,5.369999885559082,2.9600000381469727,10.210000038146973,1.1399999856948853,2.4850000143051147],
"upper":[5.090000152587891,6.257500052452087,5.480000019073486,2.5899999141693115,7.257500171661377,6.824999928474426,5.5950000286102295,3.5899999141693115,20.459999084472656,5.630000114440918,3.8499999046325684,10.770000457763672,2.119999885559082,2.84499990940094],
"x":["광양망덕","광양적량","광양초남","금강하구","낙동명지","낙동을숙","마산봉암","새만금","시화반월","영산목포","영산영암","울산매암","인천송도","천수만"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"정점명"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"&merged_groups"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","광양적량","

In [13]:
df_data.select{ 정점명 and rtmWqDoxn and rtmWqWtchDtlDt }
    .plot{
        layout {
            title = "해수 용존산소 정보"
            size = 2000 to 1000
        }
        x(rtmWqWtchDtlDt) { axis.name = "관측일시"}
        y(rtmWqDoxn) {axis.name ="용존산소"}
       // y.axis.limits = 0.0..12.0
        line{
            color(정점명){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }
        //  facetWrap(nRow = 3){ facet(gruNam)   }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="YDAFcZ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2000.0, 
 height: 1000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("YDAFcZ");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"해수 용존산소 정보"
},
"mapping":{
},
"data":{
"rtmWqDoxn":[13.866000175476074,10.5600004196167,11.529999732971191,9.0600004196167,10.989999771118164,13.289999961853027,11.989999771118164,10.210000038146973,9.65999984741211,11.149999618530273,10.199999809265137,11.84000015258789,9.6899995803833,10.59000015258789,13.807999610900879,9.649999618530273,12.039999961853027,11.529999732971191,10.210000038146973,13.67199993133545,11.170000076293945,9.6899995803833,9.65999984741211,11.199999809265137,10.210000038146973,12.130000114440918,13.706000328063965,11.25,12.210000038146973,13.73900032043457,10.1899995803833,10.579999923706055,9.699999809265137,9.8100004196167,10.753999710083008,11.510000228881836,11.279999732971191,9.630000114440918,10.350000381469727,10.77400016784668,10.1899995803833,12.149999618530273,13.72700023651123,12.149999618530273,11.229999542236328,9.800000190734863,9.9399995803833,9.6899995803833,13.697999954223633,10.170000076293945,11.350000381469727,13.489999771118164,10.180000305175781,12.100000381469727,13.708000183105469,11.180000305175781,11.300000190734863,11.180000305175781,13.711999893188477,10.180000305175781,9.819999694824219,12.09000015258789,9.369999885559082,9.869999885559082,11.289999961853027,10.1899995803833,13.696999549865723,9.850000381469727,12.170000076293945,11.270000457763672,9.3100004196167,13.470000267028809,11.399999618530273,9.90999984741211,10.180000305175781,9.829999923706055,12.029999732971191,10.793000221252441,11.279999732971191,13.670000076293945,12.210000038146973,11.260000228881836,10.1899995803833,13.460000038146973,12.199999809265137,9.949999809265137,13.869000434875488,13.0,9.329999923706055,9.859999656677246,11.149999618530273,10.170000076293945,11.069999694824219,12.0600004196167,12.960000038146973,11.109999656677246,10.170000076293945,9.819999694824219,13.90999984741211,10.781000137329102,11.350000381469727,11.069999694824219,10.180000305175781,12.100000381469727,9.449999809265137,13.069999694824219,9.779999732971191,10.289999961853027,10.390000343322754,13.934000015258789,12.15999984741211,9.819999694824219,11.0600004196167,10.786999702453613,10.180000305175781,10.781999588012695,10.149999618530273,13.923999786376953,9.579999923706055,11.050000190734863,12.149999618530273,9.789999961853027,9.800000190734863,11.09000015258789,12.100000381469727,13.920999526977539,10.130000114440918,9.760000228881836,11.149999618530273,12.770000457763672,10.100000381469727,12.15999984741211,13.92199993133545,10.460000038146973,11.350000381469727,9.470000267028809,12.119999885559082,10.300000190734863,11.180000305175781,9.760000228881836,12.789999961853027,9.699999809265137,11.109999656677246,11.529999732971191,13.906000137329102,10.718999862670898,11.8100004196167,10.15999984741211,9.800000190734863,11.25,11.079999923706055,10.130000114440918,9.760000228881836,10.722999572753906,11.739999771118164,13.914999961853027,11.59000015258789,10.170000076293945,11.140000343322754,9.699999809265137,13.895000457763672,9.770000457763672,11.699999809265137,9.279999732971191,10.5,13

In [14]:
df_data.select{ 정점명 and ph and rtmWqWtchDtlDt }
    .plot{
        layout {
            title = "해수 이온농도 정보"
            size = 2000 to 1000
        }
        x(rtmWqWtchDtlDt) { axis.name = "관측일시"}
        y(ph) {axis.name ="이온농도"}
      //  y.axis.limits = 4.0..10.0
        line{
            color(정점명){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }
        //  facetWrap(nRow = 3){ facet(gruNam)   }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="gSFFCQ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2000.0, 
 height: 1000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("gSFFCQ");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"해수 이온농도 정보"
},
"mapping":{
},
"data":{
"정점명":["영산영암","광양망덕","천수만","울산매암","시화반월","낙동명지","새만금","광양적량","영산목포","시화반월","광양적량","새만금","영산목포","광양망덕","영산영암","영산목포","새만금","천수만","광양적량","영산영암","시화반월","낙동을숙","영산목포","시화반월","광양적량","새만금","영산영암","시화반월","새만금","영산영암","광양적량","광양망덕","영산목포","낙동을숙","금강하구","천수만","시화반월","영산목포","광양망덕","금강하구","광양적량","새만금","영산영암","새만금","시화반월","낙동을숙","마산봉암","영산목포","영산영암","광양적량","천수만","낙동명지","광양적량","새만금","영산영암","시화반월","천수만","시화반월","영산영암","광양적량","영산목포","새만금","울산매암","낙동을숙","천수만","광양적량","영산영암","영산목포","새만금","시화반월","울산매암","낙동명지","시화반월","영산목포","광양적량","낙동을숙","새만금","금강하구","천수만","영산영암","새만금","시화반월","광양적량","낙동명지","새만금","마산봉암","영산영암","낙동명지","울산매암","영산목포","시화반월","광양적량","광양초남","새만금","낙동명지","시화반월","광양적량","영산목포","영산영암","금강하구","천수만","시화반월","광양적량","새만금","낙동을숙","낙동명지","영산목포","마산봉암","광양망덕","영산영암","새만금","영산목포","시화반월","금강하구","광양적량","금강하구","광양적량","영산영암","낙동을숙","시화반월","새만금","영산목포","영산목포","시화반월","새만금","영산영암","광양적량","영산목포","시화반월","낙동명지","광양적량","새만금","영산영암","광양망덕","천수만","낙동을숙","새만금","광양망덕","시화반월","영산목포","낙동명지","낙동을숙","시화반월","천수만","영산영암","금강하구","새만금","광양적량","영산목포","광양초남","시화반월","광양적량","영산목포","금강하구","새만금","영산영암","천수만","광양적량","시화반월","낙동을숙","영산영암","영산목포","새만금","울산매암","광양망덕","영산영암","영산목포","새만금","광양망덕","금강하구","영산목포","낙동명지","천수만","울산매암","광양초남","새만금","영산영암","광양적량","시화반월","광양망덕","낙동을숙","울산매암","영산목포","시화반월","새만금","광양적량","광양초남","광양망덕","영산영암","영산영암","천수만","광양초남","영산목포","광양적량","새만금","낙동을숙","시화반월","영산목포","영산영암","새만금","마산봉암","광양적량","새만금","영산목포","광양초남","금강하구","시화반월","천수만","영산영암","낙동을숙","영산목포","시화반월","새만금","영산영암","광양적량","광양초남","광양적량","천수만","시화반월","광양초남","영산목포","울산매암","새만금","영산영암","영산목포","새만금","광양초남","영산영암","광양적량","낙동을숙","금강하구","울산매암","새만금","광양적량","영산목포","영산영암","낙동명지","천수만","시화반월","광양초남","영산영암","울산매암","영산목포","금강하구","광양초남","시화반월","광양적량","마산봉암","천수만","영산영암","새만금","광양적량","영산목포","시화반월","시화반월","영산영암","광양적량","영산목포","새만금","영산목포","새만금","영산영암","광양초남","금강하구","광양적량","낙동을숙","천수만","낙동명지","시화반월","영산영암","광양적량","광양초남","시화반월","영산목포","낙동명지","광양초남","시화반월","낙동을숙","새만금","광양적량","영산영암","천수만","영산목포","시화반월","광양초남","새만금","영산영암","광양적량","영산목포","광양적량","낙동을숙","시화반월","영산영암","금강하구","마산봉암","천수만","영산목포","새만금","광양적량","영산목포","금강하구","마산봉암","새만금","영산영암","시화반월","울산매암","천수만","광양적량","낙동명지","영산목포","마산봉암","낙동을숙","새만금","시화반월","영산영암","광양적량","시화반월","영산영암","영산목포","마산봉암","낙동을숙","새만금","천수만","마산봉암","낙동을숙","영산목포","금강하구","광양적량","영산영암","시화반월","새만금","광양망덕","영산목포","금강하구","광양망덕","시화반월","광양적량","영산영암","새만금","천수만","영산목포","영산영암","광양적량","광양망덕","낙동을숙","새만금","영산목포","영산영암","새만금","광양망덕","광양적량","새만금","영산목포","광양적량","마산봉암","낙동을숙","시화반월","영산영암","천수만","영산목포","새만금","영산영암","시화반월","광양적량","마산봉암","낙동을숙","영산영암","천수만","금강하구","광양망덕","광양적량","울산매암","새만금","영산목포","시화반월","새만금","시화반월","영산영암","광양적량","광양망덕","영산목포","울산매암","금강하구","낙동을숙","울산매암","시화반월","광양망덕","금강하구","영산영암","영산목포","천수만","새만금","광양적량","울산매암","새만금","영산영암","시화반월","광양망덕","광양적량","영산영암","울산매암","새만금","시화반월","광양적량","금강하구","마산봉암","낙동을숙","천수만","영산목포","낙동명지","광양초남","금강하구","영산목포","시화반월","광양초남","낙동명지","영산영암","울산매암","마산봉암","새만금","광양적량","울산매암","천수만","시화반월","낙동을숙","영산영암","새만금","금강하구","영산목포","광양적량","낙동명지","금강하구","시화반월","새만금","울산매암","광양적량","영산영암","시화반월","영산영암","울산매암","광양적량","